In [1]:
# Install required packages
!pip install boto3 pandas numpy scikit-learn sentence-transformers datarec-lib requests

<string>:1: UserWarning: datarec-lib has been renamed to datarec. Please update your dependencies.


In [2]:
# Importing needed packages
import os
import zipfile
import pandas as pd
import requests
import numpy as np
import requests
from io import BytesIO

import boto3
from botocore import UNSIGNED
from botocore.config import Config
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from transformers import BertTokenizer, BertModel
import torch

In [3]:
# Task 1 - Check if files exist
BUCKET = "ynf0058-de300-lab3"
KEY = "ml-1m/ratings.dat" 

s3 = boto3.client(
    "s3",
    region_name="us-east-1",
    config=Config(signature_version=UNSIGNED)
)

def file_exists_public(bucket, key):
    try:
        s3.head_object(Bucket=bucket, Key=key)
        return True
    except:
        return False

print(file_exists_public(BUCKET, KEY))

True


In [4]:
# Task 2 - Creating the embeddings
## Starting with reading in the dataset
url_for_movies = "https://ynf0058-de300-lab3.s3.amazonaws.com/ml-1m/movies.dat"
def load_movies():
    movies = pd.read_csv(url_for_movies, sep= "::", engine = "python", names = ["MovieID", "Title", "Genres"], encoding = "latin-1")
    return movies
movies = load_movies()
movies.head()

,MovieID,Title,Genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [5]:
# Creates the specific text to be used by the BERT algorithm
def create_bert_text(movies):
    movies = movies.copy()
    movies["bert_text"] = ("Movie title: " + movies["Title"] + ". Genres: " + movies["Genres"])
    return movies

movies = create_bert_text(movies)
movies[["MovieID", "Title", "Genres", "bert_text"]].head()

,MovieID,Title,Genres,bert_text
0,1,Toy Story (1995),Animation|Children's|Comedy,Movie title: Toy Story (1995). Genres: Animati...
1,2,Jumanji (1995),Adventure|Children's|Fantasy,Movie title: Jumanji (1995). Genres: Adventure...
2,3,Grumpier Old Men (1995),Comedy|Romance,Movie title: Grumpier Old Men (1995). Genres: ...
3,4,Waiting to Exhale (1995),Comedy|Drama,Movie title: Waiting to Exhale (1995). Genres:...
4,5,Father of the Bride Part II (1995),Comedy,Movie title: Father of the Bride Part II (1995...


In [6]:
def create_raw_bert_embeddings(movies):
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    model = BertModel.from_pretrained("bert-base-uncased")
    model.eval()
    texts = movies["bert_text"].tolist()
    embeddings = []
    for text in texts:
        inputs = tokenizer(text, return_tensors = "pt", truncation = True, max_length = 128)
        with torch.no_grad():
            outputs = model(**inputs)

        cls_embedding = outputs.last_hidden_state[:, 0, :]
        embeddings.append(cls_embedding.squeeze().numpy())

    return np.array(embeddings)
raw_bert_embeddings = create_raw_bert_embeddings(movies)
raw_bert_embeddings.shape

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(3883, 768)

In [7]:
import os

os.makedirs("outputs", exist_ok=True)

movies.to_csv(
    "outputs/movies_full.csv",
    index=False
)

np.save(
    "outputs/movie_embeddings_full.npy",
    raw_bert_embeddings
)

os.listdir("outputs")

['movies_full.csv', 'movie_embeddings_full.npy']

In [8]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np
import os

BUCKET = "ynf0058-de300-lab3"

ratings_url = f"https://{BUCKET}.s3.amazonaws.com/ml-1m/ratings.dat"

ratings = pd.read_csv(
    ratings_url,
    sep="::",
    engine="python",
    names=["UserID", "MovieID", "Rating", "Timestamp"],
    encoding="latin-1"
)

ratings["Datetime"] = pd.to_datetime(ratings["Timestamp"], unit="s")

ratings.head()

,UserID,MovieID,Rating,Timestamp,Datetime
0,1,1193,5,978300760,2000-12-31 22:12:40
1,1,661,3,978302109,2000-12-31 22:35:09
2,1,914,3,978301968,2000-12-31 22:32:48
3,1,3408,4,978300275,2000-12-31 22:04:35
4,1,2355,5,978824291,2001-01-06 23:38:11


In [9]:
print(ratings["Datetime"].min())
print(ratings["Datetime"].max())
print(ratings.shape)

2000-04-25 23:05:32
2003-02-28 17:49:50
(1000209, 5)


In [10]:
part1 = ratings[
    (ratings["Datetime"] >= "2000-04-25") &
    (ratings["Datetime"] <= "2000-08-03")
].copy()

part2 = ratings[
    (ratings["Datetime"] >= "2000-08-04") &
    (ratings["Datetime"] <= "2000-10-31")
].copy()

part3 = ratings[
    (ratings["Datetime"] >= "2000-11-01") &
    (ratings["Datetime"] < "2000-11-26")
].copy()

part4 = ratings[
    ratings["Datetime"] >= "2000-11-26"
].copy()

partitions = [part1, part2, part3, part4]

for i, part in enumerate(partitions, start=1):
    print(f"Part {i}: {part.shape[0]} ratings")
    print(part["Datetime"].min(), "to", part["Datetime"].max())

Part 1: 244635 ratings
2000-04-25 23:05:32 to 2000-08-02 23:59:52
Part 2: 233929 ratings
2000-08-04 00:00:39 to 2000-10-30 23:45:22
Part 3: 245465 ratings
2000-11-01 00:21:22 to 2000-11-25 23:59:51
Part 4: 254267 ratings
2000-11-26 00:00:51 to 2003-02-28 17:49:50


In [11]:
movies_full = pd.read_csv("outputs/movies_full.csv")
movie_embeddings_full = np.load("outputs/movie_embeddings_full.npy")
print(movies_full.shape)
print(movie_embeddings_full.shape)
movies_full.head()

(3883, 4)
(3883, 768)


,MovieID,Title,Genres,bert_text
0,1,Toy Story (1995),Animation|Children's|Comedy,Movie title: Toy Story (1995). Genres: Animati...
1,2,Jumanji (1995),Adventure|Children's|Fantasy,Movie title: Jumanji (1995). Genres: Adventure...
2,3,Grumpier Old Men (1995),Comedy|Romance,Movie title: Grumpier Old Men (1995). Genres: ...
3,4,Waiting to Exhale (1995),Comedy|Drama,Movie title: Waiting to Exhale (1995). Genres:...
4,5,Father of the Bride Part II (1995),Comedy,Movie title: Father of the Bride Part II (1995...


In [12]:
if "Year" not in movies_full.columns:
    movies_full["Year"] = movies_full["Title"].str.extract(r"\((\d{4})\)")

movies_full.head()

,MovieID,Title,Genres,bert_text,Year
0,1,Toy Story (1995),Animation|Children's|Comedy,Movie title: Toy Story (1995). Genres: Animati...,1995
1,2,Jumanji (1995),Adventure|Children's|Fantasy,Movie title: Jumanji (1995). Genres: Adventure...,1995
2,3,Grumpier Old Men (1995),Comedy|Romance,Movie title: Grumpier Old Men (1995). Genres: ...,1995
3,4,Waiting to Exhale (1995),Comedy|Drama,Movie title: Waiting to Exhale (1995). Genres:...,1995
4,5,Father of the Bride Part II (1995),Comedy,Movie title: Father of the Bride Part II (1995...,1995


In [13]:
# Defining a cold user's recommendations
def cold_user_recommendation(movies, embeddings, top_k = 5):
    average_embedding = embeddings.mean(axis = 0).reshape(1, -1)
    similarity = cosine_similarity(average_embedding, embeddings)[0]
    top_indices = similarity.argsort()[-top_k:][::-1]
    recs = movies.iloc[top_indices][["MovieID", "Title", "Genres", "Year"]].copy()
    recs["Similarity"] = similarity[top_indices]
    return recs
# Selecting the top user
def select_top_user(ratings):
    count_per_user = ratings.groupby("UserID").size().reset_index(name = "Interaction_Count")
    upper_threshold = count_per_user["Interaction_Count"].quantile(0.95)
    top_users = count_per_user[count_per_user["Interaction_Count"] >= upper_threshold]
    chosen_user = top_users.sample(1, random_state = 42).iloc[0]
    return int(chosen_user["UserID"]), int(chosen_user["Interaction_Count"])
# Defining recommendations for the top user
def top_user_recommendation(user_id, ratings, movies, embeddings, top_k = 5, min_rating = 4):
    movie_id_to_index = {movie_id: i for i, movie_id in enumerate(movies["MovieID"])}
    user_ratings = ratings[(ratings["UserID"] == user_id) & (ratings["Rating"] >= min_rating)].copy()
    user_ratings = user_ratings[user_ratings["MovieID"].isin(movie_id_to_index.keys())]
    rated_indices = [movie_id_to_index[movie_id] for movie_id in user_ratings["MovieID"]]
    user_embedding = embeddings[rated_indices].mean(axis = 0).reshape(1, -1)
    similarity = cosine_similarity(user_embedding, embeddings)[0]
    already_rated = set(user_ratings["MovieID"])
    candidate_indices = [i for i, movie_id in enumerate(movies["MovieID"]) if movie_id not in already_rated]
    ranked_indices = sorted(candidate_indices, key = lambda i: similarity[i], reverse = True)
    top_indices = ranked_indices[: top_k]
    recs = movies.iloc[top_indices][["MovieID", "Title", "Genres", "Year"]].copy()
    recs["Similarity"] = similarity[top_indices]
    return recs

In [14]:
import os

os.makedirs("outputs/recommendations", exist_ok=True)

simulated_hours = [0, 10, 20, 30]
observed_ratings = pd.DataFrame()

for i, part in enumerate(partitions, start=1):
    simulated_hour = simulated_hours[i - 1]

    observed_ratings = pd.concat(
        [observed_ratings, part],
        ignore_index=True
    )

    sampled_users = pd.Series(observed_ratings["UserID"].unique()).sample(
        frac=0.30,
        random_state=42 + i
    )

    sampled_ratings = observed_ratings[
        observed_ratings["UserID"].isin(sampled_users)
    ].copy()

    cold_recs = cold_user_recommendation(
        movies_full,
        movie_embeddings_full,
        top_k=5
    )

    top_user_id, top_user_interactions = select_top_user(sampled_ratings)

    top_recs = top_user_recommendation(
        top_user_id,
        observed_ratings,
        movies_full,
        movie_embeddings_full,
        top_k=5
    )

    top_user_history = observed_ratings[
        observed_ratings["UserID"] == top_user_id
    ]

    output_rows = []

    for _, row in cold_recs.iterrows():
        output_rows.append({
            "Iteration": i,
            "Simulated_Hour": simulated_hour,
            "User_Type": "Cold User",
            "User_ID": None,
            "Last_Interaction_Time": None,
            "Number_Of_Ratings_Observed": 0,
            "Recommended_MovieID": row["MovieID"],
            "Recommended_Title": row["Title"],
            "Recommended_Genres": row["Genres"],
            "Similarity": row["Similarity"]
        })

    for _, row in top_recs.iterrows():
        output_rows.append({
            "Iteration": i,
            "Simulated_Hour": simulated_hour,
            "User_Type": "Top User",
            "User_ID": top_user_id,
            "Last_Interaction_Time": top_user_history["Datetime"].max(),
            "Number_Of_Ratings_Observed": len(top_user_history),
            "Recommended_MovieID": row["MovieID"],
            "Recommended_Title": row["Title"],
            "Recommended_Genres": row["Genres"],
            "Similarity": row["Similarity"]
        })

    results = pd.DataFrame(output_rows)

    output_path = f"outputs/recommendations/recommendations_iteration_{i}_hour_{simulated_hour}.csv"
    results.to_csv(output_path, index=False)

    print(f"Saved: {output_path}")
    print(results)

Saved: outputs/recommendations/recommendations_iteration_1_hour_0.csv
   Iteration  Simulated_Hour  User_Type  User_ID Last_Interaction_Time  \
0          1               0  Cold User      NaN                   NaT   
1          1               0  Cold User      NaN                   NaT   
2          1               0  Cold User      NaN                   NaT   
3          1               0  Cold User      NaN                   NaT   
4          1               0  Cold User      NaN                   NaT   
5          1               0   Top User   4884.0   2000-07-08 08:56:20   
6          1               0   Top User   4884.0   2000-07-08 08:56:20   
7          1               0   Top User   4884.0   2000-07-08 08:56:20   
8          1               0   Top User   4884.0   2000-07-08 08:56:20   
9          1               0   Top User   4884.0   2000-07-08 08:56:20   

   Number_Of_Ratings_Observed  Recommended_MovieID         Recommended_Title  \
0                           0      

In [15]:
os.listdir("outputs/recommendations")

['recommendations_iteration_1_hour_0.csv',
 'recommendations_iteration_2_hour_10.csv',
 'recommendations_iteration_3_hour_20.csv',
 'recommendations_iteration_4_hour_30.csv']